In [1]:
import numpy as np
import pandas
import pm4py
from matplotlib import pyplot as plt
from sklearn.mixture import GaussianMixture
import scipy.stats as stats
import ot
import os
from tqdm import tqdm
import collections
import matplotlib.dates as md
import importlib
import pickle
import random
import math
import CRPS.CRPS as pscore
import datetime

import multiprocessing as mp
mp.set_start_method('spawn')


np.seterr(all='warn', over='ignore')
pandas.set_option('display.max_columns', None)
#pandas.set_option('display.max_rows', None)


import sys
sys.path.append('../../TaskExecutionTimeMining/')
from drbart_parser import *
from event_log_transformer import *

#sys.path.append('../../Evaluation')
sys.path.append('../../Evaluation/')
from conduct_evaluation import ConductEvaluation
from normal_evaluation.drbart_evaluation import *

get_pscores = lambda likelihoods : [pscore(likelihoods[1][i], likelihoods[2][k][3]).compute()[0] for i, k in enumerate(list(likelihoods[0].keys()))]

In [2]:
#model_name = 'bpic_2017_all_2'
model_name = 'artificial'

n_processes = 32
batch_size = 50

log_name = 'test'

with open('../transformed_event_logs/artificial_start_end_2_'+log_name+'.pickle', 'rb') as f:
    test_event_log = pickle.load(f)

test_event_log['case:concept:name'] = test_event_log['case:concept:name'].astype(str)
known_resources = ['1', 'Clark', 'Jane', 'Joe', 'Karsten']
known_activities = ['DIAGNOSIS', 'QUALITY_CONTROL', 'REPAIR']

In [3]:
N = 1000
import conduct_evaluation
get_pscores = lambda likelihoods : [pscore(likelihoods[1][i], likelihoods[2][k][3]).compute()[0] for i, k in enumerate(list(likelihoods[0].keys()))]

likelihoods_A = None
likelihoods_R = None
likelihoods_R_A = None
likelihoods_R_A_S = None
likelihoods_R_A_S_AC = None
likelihoods_R_A_S_RC = None
likelihoods_R_A_S_RC_AC = None
likelihoods_R_A_S_RC_AC_V = None
likelihoods_R_A_S_D = None
likelihoods_R_A_S_D_RC_AC = None

In [4]:
drbart_model_A = DRBART(parser_dir = '../../../models/standard_numerical_stable/'+model_name+'/concept-name/',
                     strict_parser=False)
evaluator_A = conduct_evaluation.ConductEvaluation(drbart_model_A, SampleOutcomes_DRBART_Normal_A, {
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource',
                                                    },
                                     test_event_log, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True)

100%|██████████| 892/892 [00:04<00:00, 190.84it/s]


In [5]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-0.007668547338179578332279800872')

In [6]:
np.mean(get_pscores(likelihoods_A))

np.float64(12481.449622245726)

In [7]:
drbart_model_R_A = DRBART(parser_dir = '../../../models/standard_numerical_stable/'+model_name+'/concept-name_resource/',
                     strict_parser=False)
evaluator_R_A = conduct_evaluation.ConductEvaluation(drbart_model_R_A, SampleOutcomes_DRBART_Normal_R_A,
                                                   {
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource',
                                                    },
                                     test_event_log, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_R_A = evaluator_R_A.sample_cases(False, True)

100%|██████████| 892/892 [00:05<00:00, 172.16it/s]


In [8]:
np.mean([v.ln() for v in likelihoods_R_A[0].values()])

Decimal('-0.04359602228499098432771157618')

In [9]:
np.mean(get_pscores(likelihoods_R_A))

np.float64(12285.44966337041)

In [10]:
drbart_model_R = DRBART(parser_dir = '../../../models/standard_numerical_stable/'+model_name+'/resource/',
                     strict_parser=False)
evaluator_R = conduct_evaluation.ConductEvaluation(drbart_model_R, SampleOutcomes_DRBART_Normal_R,
                                                   {
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource',
                                                    },
                                     test_event_log, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_R = evaluator_R.sample_cases(False, True)

100%|██████████| 892/892 [00:05<00:00, 166.42it/s]


In [11]:
np.mean([v.ln() for v in likelihoods_R[0].values()])

Decimal('-0.2737384901567002252756809543')

In [12]:
np.mean(get_pscores(likelihoods_R))

np.float64(20111.28665537727)

In [13]:
drbart_model_R_A_S = DRBART(parser_dir = '../../../models/standard_numerical_stable/'+model_name+'/concept-name_resource_seconds-in-day/',
                     strict_parser=False)
evaluator_R_A_S = conduct_evaluation.ConductEvaluation(drbart_model_R_A_S, SampleOutcomes_DRBART_Normal_R_A_S,
                                                   {
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource',
                                                    },
                                     test_event_log, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_R_A_S = evaluator_R_A_S.sample_cases(False, True)

100%|██████████| 892/892 [00:05<00:00, 166.87it/s]


In [14]:
np.mean([v.ln() for v in likelihoods_R_A_S[0].values()])

Decimal('-0.03859819969368651766090220139')

In [15]:
np.mean(get_pscores(likelihoods_R_A_S))

np.float64(13134.737363628019)

In [16]:
drbart_model_R_A_S_AC = DRBART(parser_dir = '../../../models/standard_numerical_stable/'+model_name+'/concept-name_resource_seconds-in-day_activity-count/',
                     strict_parser=False)
evaluator_R_A_S_AC = conduct_evaluation.ConductEvaluation(drbart_model_R_A_S_AC, SampleOutcomes_DRBART_Normal_R_A_S_AC,
                                                   {
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource',
                                                        'known_activities' : known_activities,
                                                    },
                                     test_event_log, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_R_A_S_AC = evaluator_R_A_S_AC.sample_cases(False, True)

100%|██████████| 892/892 [00:04<00:00, 187.87it/s]


In [17]:
np.mean([v.ln() for v in likelihoods_R_A_S_AC[0].values()])

Decimal('-0.4797052669414775236161907187')

In [18]:
np.mean(get_pscores(likelihoods_R_A_S_AC))

np.float64(11336.666605083878)

In [19]:
drbart_model_R_A_S_RC = DRBART(parser_dir = '../../../models/standard_numerical_stable/'+model_name+'/concept-name_resource_seconds-in-day_resource-count/',
                     strict_parser=False)
evaluator_R_A_S_RC = conduct_evaluation.ConductEvaluation(drbart_model_R_A_S_RC, SampleOutcomes_DRBART_Normal_R_A_S_RC,
                                                   {
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource',
                                                        'known_resources' : known_resources
                                                    },
                                     test_event_log, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_R_A_S_RC = evaluator_R_A_S_RC.sample_cases(False, True)

100%|██████████| 892/892 [00:04<00:00, 190.69it/s]


In [20]:
np.mean([v.ln() for v in likelihoods_R_A_S_RC[0].values()])

Decimal('-0.1479507443846908673259598577')

In [21]:
np.mean(get_pscores(likelihoods_R_A_S_RC))

np.float64(11857.440995753092)

In [22]:
drbart_model_R_A_S_RC_AC = DRBART(parser_dir = '../../../models/standard_numerical_stable/'+model_name+'/concept-name_resource_seconds-in-day_activity-count_resource-count/',
                     strict_parser=False)
evaluator_R_A_S_RC_AC = conduct_evaluation.ConductEvaluation(drbart_model_R_A_S_RC_AC, SampleOutcomes_DRBART_Normal_R_A_S_RC_AC,
                                                   {
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource',
                                                        'known_resources' : known_resources,
                                                        'known_activities' : known_activities,
                                                    },
                                     test_event_log, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_R_A_S_RC_AC = evaluator_R_A_S_RC_AC.sample_cases(False, True)

100%|██████████| 892/892 [00:04<00:00, 190.24it/s]


In [23]:
np.mean([v.ln() for v in likelihoods_R_A_S_RC_AC[0].values()])

Decimal('-0.2512845402193366313260788557')

In [24]:
np.mean(get_pscores(likelihoods_R_A_S_RC_AC))

np.float64(11435.549857090775)

In [27]:
drbart_model_R_A_S_D = DRBART(parser_dir = '../../../models/standard_numerical_stable/'+model_name+'/concept-name_resource_seconds-in-day_day-of-week/',
                     strict_parser=False)
evaluator_R_A_S_D = conduct_evaluation.ConductEvaluation(drbart_model_R_A_S_D, SampleOutcomes_DRBART_Normal_R_A_S_D,
                                                   {
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource',
                                                    },
                                     test_event_log, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_R_A_S_D = evaluator_R_A_S_D.sample_cases(False, True)

100%|██████████| 892/892 [00:04<00:00, 179.52it/s]


In [28]:
np.mean([v.ln() for v in likelihoods_R_A_S_D[0].values()])

Decimal('-0.06720336131330672339414668843')

In [29]:
np.mean(get_pscores(likelihoods_R_A_S_D))

np.float64(11111.266382665139)

In [30]:
drbart_model_R_A_S_D_RC_AC = DRBART(parser_dir = '../../../models/standard_numerical_stable/'+model_name+'/concept-name_resource_seconds-in-day_day-of-week_activity-count_resoure-count/',
                     strict_parser=False)
evaluator_R_A_S_D_RC_AC = conduct_evaluation.ConductEvaluation(drbart_model_R_A_S_D_RC_AC, SampleOutcomes_DRBART_Normal_R_A_S_D_RC_AC,
                                                   {
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource',
                                                        'known_resources' : known_resources,
                                                        'known_activities' : known_activities,
                                                    },
                                     test_event_log, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_R_A_S_D_RC_AC = evaluator_R_A_S_D_RC_AC.sample_cases(False, True)

100%|██████████| 892/892 [00:04<00:00, 181.89it/s]


In [31]:
np.mean([v.ln() for v in likelihoods_R_A_S_D_RC_AC[0].values()])

Decimal('-0.1003763806882048891196693964')

In [32]:
np.mean(get_pscores(likelihoods_R_A_S_D_RC_AC))

np.float64(14743.37200632238)

In [ ]:
results = {
    'drbart_model_A' : likelihoods_A,
    'drbart_model_R' : likelihoods_R,
    'drbart_model_R_A' : likelihoods_R_A,
    'drbart_model_R_A_S' : likelihoods_R_A_S,
    'drbart_model_R_A_S_AC' : likelihoods_R_A_S_AC,
    'drbart_model_R_A_S_RC' : likelihoods_R_A_S_RC,
    'drbart_model_R_A_S_RC_AC' : likelihoods_R_A_S_RC_AC,
    'drbart_model_R_A_S_RC_AC_V' : likelihoods_R_A_S_RC_AC_V,
    'drbart_model_R_A_S_D' : likelihoods_R_A_S_D,
    'drbart_model_R_A_S_D_RC_CC' : likelihoods_R_A_S_D_RC_AC
}
with open('./'+model_name+'_dr_bart_evaluation_'+log_name+'.pickle', 'wb') as handle:
    pickle.dump(results, handle)